# Séance 8 — Suivi des expériences et registre de modèles (MLflow)
*Session 8 — Experiment tracking and model registry (MLflow)*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/08_mlflow_registre.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ● séance complète |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ● séance complète |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | — hors parcours |

**À la fin de cette séance, vous saurez :**

1. retrouver les exécutions, paramètres et métriques d'un entraînement ;
2. journaliser sa propre expérience (paramètres, métriques, modèle) ;
3. enregistrer une version de modèle et lui attribuer un **alias** (champion, challenger) ;
4. expliquer pourquoi le service charge un modèle par alias et non par fichier.

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

In [ ]:
cfg = preparer("entrainer")
import mlflow
from fraude360.modeling import registre
client = registre.configurer(cfg)                   # MLflow local (SQLite) : même API que Databricks
print("Suivi :", mlflow.get_tracking_uri())
runs = mlflow.search_runs(experiment_names=["fraude360"])
cols = [c for c in runs.columns if c.startswith(("tags.mlflow.runName", "metrics.pr_auc", "params.modele"))]
runs[["run_id", "start_time"] + cols].head(8)

In [ ]:
versions = client.search_model_versions(f"name='{registre.NOM_MODELE}'")
for v in versions:
    print("version", v.version, "| alias :", v.aliases, "| run :", v.run_id[:8])
print("Champion actuel : version", registre.version_par_alias("champion", cfg))

**Alias plutôt que fichier.** Le service de scoring charge `models:/fraude360_detection@champion`. Promouvoir un nouveau modèle
= déplacer l'alias, sans redéployer le code; revenir en arrière = remettre l'alias sur l'ancienne version. Toute promotion est
tracée (qui, quand, quelle version), ce qu'exigent les auditeurs (ligne directrice E-23 du BSIF).

In [ ]:
from fraude360.modeling import donnees, entrainement
jeux = donnees.decouper(cfg)
num, cat = jeux.variables_numeriques, jeux.variables_categorielles
Xtr, ytr = jeux.entrainement[num + cat], jeux.entrainement["est_fraude"].values
Xva, yva = jeux.validation[num + cat], jeux.validation["est_fraude"].values
# une expérience séparée, pour ne pas mélanger vos essais avec le pipeline
nom_exp = "cours_etudiant"
if client.get_experiment_by_name(nom_exp) is None:
    client.create_experiment(nom_exp, artifact_location=(cfg.chemin("mlruns") / "artefacts_etudiant").resolve().as_uri())
mlflow.set_experiment(nom_exp)

### Exercice 8.1 — 🟣 Gestion + Informatique

Dans un `with mlflow.start_run(run_name="lr_C_0.1") as run:`, entraînez une régression logistique
(`entrainement.construire_modeles(...)["regression_logistique"]` avec `modele__C=0.1` via `set_params`), journalisez le paramètre
`C`, la métrique `pr_auc` (validation) et le modèle (`mlflow.sklearn.log_model(pipe, name="modele", serialization_format="cloudpickle")`). Gardez `run_id = run.info.run_id`
et l'URI du modèle dans `uri_modele` (attribut `model_uri` de l'objet renvoyé par `log_model`).

In [ ]:
# VOTRE CODE ICI
from sklearn.metrics import average_precision_score
run_id, uri_modele = ..., ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_r = mlflow.get_run(run_id)
verifier("pr_auc" in _r.data.metrics and _r.data.params.get("C") == "0.1", "Paramètre C et métrique pr_auc journalisés")

### Exercice 8.2 — 🔷 Informatique — approfondissement

Enregistrez ce modèle sous le nom `fraude360_etudiant` (`mlflow.register_model`), posez-lui l'alias `candidat`
(`client.set_registered_model_alias`), puis rechargez-le **par alias** (`mlflow.sklearn.load_model("models:/fraude360_etudiant@candidat")`)
dans `recharge`.

In [ ]:
# VOTRE CODE ICI
recharge = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_orig = mlflow.sklearn.load_model(uri_modele)
verifier(np.allclose(recharge.predict_proba(Xva.head(50))[:, 1], _orig.predict_proba(Xva.head(50))[:, 1]),
         "Le modèle rechargé par alias doit prédire comme l'original")

In [ ]:
mlflow.set_experiment("fraude360")   # on rend la main au pipeline

**❓ Question (🔶 Gestion — décision d'affaires) —** Qui devrait avoir le droit de déplacer l'alias `champion` ? Décrivez un processus à « quatre yeux » compatible avec les trois lignes de défense.

*Votre réponse ici.*

## Bilan

- MLflow garde la trace de chaque essai : paramètres, métriques, artefacts, code.
- Le registre versionne les modèles; les alias découplent le déploiement du code.
- Le même code fonctionne en local (SQLite) et sur Databricks (Unity Catalog).

**Pour aller plus loin :**

- `src/fraude360/modeling/registre.py`; interface : `mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db`.

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*